# 🤖 TP : construire une boucle d'outils sûre et testable

**Durée indicative : 3 h — Simulation locale, sans clé API ni appel à un LLM.**

## Objectifs observables

1. séparer politique de décision, registre d'outils, mémoire et orchestration ;
2. décrire les outils avec des arguments structurés ;
3. valider types, erreurs, budgets et autorisations hors du modèle ;
4. exécuter une politique déterministe qui réagit réellement aux observations ;
5. identifier ce qu'il faudrait encore évaluer avant de brancher un LLM.

> **Limite essentielle** : le notebook n'implémente pas un agent autonome fondé sur un LLM. Une politique Python déterministe remplace volontairement le modèle afin de tester l'orchestrateur sans coût, réseau ni clé secrète. Un LLM pourrait ensuite proposer les mêmes objets `Decision`, sans recevoir davantage de privilèges.


## 1. Architecture et frontière de confiance

```text
objectif -> politique de décision -> ToolCall structuré
                                  |
                                  v
        contrôles déterministes -> outil autorisé
                                  |
                                  v
                         observation journalisée
```

La politique peut être probabiliste, mais le programme hôte garde le contrôle des schémas, autorisations, budgets, erreurs et effets de bord.


In [1]:
from dataclasses import dataclass
from datetime import datetime
import inspect
from typing import Any, Callable, get_type_hints

print("Environnement d'orchestration prêt.")


Environnement d'orchestration prêt.


## 2. Résultats d'outils et registre typé

Un outil ne renvoie pas une chaîne ambiguë mais un résultat explicite : succès, valeur ou erreur. Le registre vérifie l'existence de l'outil, le niveau de risque, la signature et les types simples avant exécution.


In [2]:
@dataclass
class ToolResult:
    ok: bool
    value: Any = None
    error: str | None = None


@dataclass
class ToolSpec:
    function: Callable
    description: str
    risk: str
    schema: dict[str, Any]


class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, ToolSpec] = {}

    def register(self, *, risk: str = "read"):
        if risk not in {"read", "high"}:
            raise ValueError("Le risque doit être 'read' ou 'high'.")

        def decorator(function: Callable):
            signature = inspect.signature(function)
            hints = get_type_hints(function)
            json_types = {str: "string", int: "integer", float: "number", bool: "boolean"}
            properties = {}
            required = []
            for name, parameter in signature.parameters.items():
                annotation = hints.get(name, str)
                properties[name] = {"type": json_types.get(annotation, "string")}
                if parameter.default is inspect.Parameter.empty:
                    required.append(name)

            self._tools[function.__name__] = ToolSpec(
                function=function,
                description=(function.__doc__ or "").strip(),
                risk=risk,
                schema={
                    "type": "object",
                    "properties": properties,
                    "required": required,
                    "additionalProperties": False,
                },
            )
            return function

        return decorator

    def documentation(self) -> list[dict[str, Any]]:
        return [
            {
                "name": name,
                "description": spec.description,
                "risk": spec.risk,
                "parameters": spec.schema,
            }
            for name, spec in self._tools.items()
        ]

    def execute(
        self,
        tool_name: str,
        arguments: dict[str, Any],
        *,
        approved: bool = False,
    ) -> ToolResult:
        spec = self._tools.get(tool_name)
        if spec is None:
            return ToolResult(False, error=f"UNKNOWN_TOOL: {tool_name}")
        if spec.risk == "high" and not approved:
            return ToolResult(False, error="APPROVAL_REQUIRED")

        try:
            signature = inspect.signature(spec.function)
            bound = signature.bind(**arguments)
            bound.apply_defaults()
        except TypeError as exc:
            return ToolResult(False, error=f"INVALID_ARGUMENTS: {exc}")

        hints = get_type_hints(spec.function)
        for name, value in bound.arguments.items():
            expected = hints.get(name)
            valid = (
                expected is None
                or isinstance(value, expected)
                or (expected is float and isinstance(value, int))
            )
            if not valid:
                return ToolResult(
                    False,
                    error=f"INVALID_TYPE: {name} doit être {expected.__name__}",
                )

        try:
            return ToolResult(True, value=spec.function(*bound.args, **bound.kwargs))
        except Exception as exc:
            return ToolResult(False, error=f"TOOL_ERROR: {exc}")


registry = ToolRegistry()


## 3. Outils locaux et déterministes

Les taux de change ci-dessous sont des constantes pédagogiques, pas des données financières actuelles. La calculatrice reçoit des champs séparés : elle n'utilise ni `eval` ni exécution de code arbitraire.


In [3]:
@registry.register(risk="read")
def recherche_base_connaissances(requete: str) -> dict:
    '''Retourne un fait structuré depuis une petite base locale.'''
    knowledge_base = {
        "chiffre_affaires_2023": {
            "montant": 4_500_000.0,
            "devise": "EUR",
            "source": "rapport_annuel_2023",
        },
        "effectif_total": {
            "effectif": 120,
            "source": "registre_rh_exemple",
        },
    }
    if requete not in knowledge_base:
        raise KeyError(f"Document introuvable : {requete}")
    return knowledge_base[requete]


@registry.register(risk="read")
def convertir_devise(montant: float, devise_source: str, devise_cible: str) -> dict:
    '''Convertit selon des taux fixes utilisés uniquement pour ce TP.'''
    taux_vers_eur = {"EUR": 1.0, "USD": 0.92, "GBP": 1.17, "JPY": 0.0062}
    source = devise_source.upper()
    cible = devise_cible.upper()
    if source not in taux_vers_eur or cible not in taux_vers_eur:
        raise ValueError("Devise non prise en charge.")
    valeur_eur = montant * taux_vers_eur[source]
    return {
        "montant": valeur_eur / taux_vers_eur[cible],
        "devise": cible,
        "taux": "constantes_pedagogiques",
    }


@registry.register(risk="read")
def calculatrice(a: float, operation: str, b: float) -> float:
    '''Applique une opération arithmétique autorisée à deux nombres.'''
    operations = {
        "+": lambda left, right: left + right,
        "-": lambda left, right: left - right,
        "*": lambda left, right: left * right,
        "/": lambda left, right: left / right,
    }
    if operation not in operations:
        raise ValueError("Opération non autorisée.")
    if operation == "/" and b == 0:
        raise ZeroDivisionError("Division par zéro.")
    return operations[operation](a, b)


for tool_documentation in registry.documentation():
    print(tool_documentation)


{'name': 'recherche_base_connaissances', 'description': 'Retourne un fait structuré depuis une petite base locale.', 'risk': 'read', 'parameters': {'type': 'object', 'properties': {'requete': {'type': 'string'}}, 'required': ['requete'], 'additionalProperties': False}}
{'name': 'convertir_devise', 'description': 'Convertit selon des taux fixes utilisés uniquement pour ce TP.', 'risk': 'read', 'parameters': {'type': 'object', 'properties': {'montant': {'type': 'number'}, 'devise_source': {'type': 'string'}, 'devise_cible': {'type': 'string'}}, 'required': ['montant', 'devise_source', 'devise_cible'], 'additionalProperties': False}}
{'name': 'calculatrice', 'description': 'Applique une opération arithmétique autorisée à deux nombres.', 'risk': 'read', 'parameters': {'type': 'object', 'properties': {'a': {'type': 'number'}, 'operation': {'type': 'string'}, 'b': {'type': 'number'}}, 'required': ['a', 'operation', 'b'], 'additionalProperties': False}}


## 4. Mémoire, décisions structurées et boucle bornée

La mémoire conserve des événements auditables. Une décision contient un résumé bref, un appel structuré ou une réponse finale ; elle n'exige pas l'affichage d'un raisonnement interne détaillé.


In [4]:
@dataclass
class Decision:
    kind: str
    summary: str
    call_id: str | None = None
    tool_name: str | None = None
    arguments: dict[str, Any] | None = None
    answer: str | None = None


class AgentMemory:
    def __init__(self):
        self.events: list[dict[str, Any]] = []

    def add(self, kind: str, content: Any) -> None:
        self.events.append({
            "time": datetime.now().isoformat(timespec="seconds"),
            "kind": kind,
            "content": content,
        })


class ToolLoop:
    def __init__(self, registry: ToolRegistry, max_steps: int = 6):
        self.registry = registry
        self.max_steps = max_steps

    def run(self, goal: str, policy: Callable):
        memory = AgentMemory()
        state: dict[str, ToolResult] = {}
        memory.add("goal", goal)

        for step in range(1, self.max_steps + 1):
            decision: Decision = policy(state)
            memory.add("decision", decision.summary)

            if decision.kind == "final":
                memory.add("final", decision.answer)
                return decision.answer, memory, state
            if decision.kind != "tool_call":
                raise ValueError(f"Type de décision inconnu : {decision.kind}")
            if not all([decision.call_id, decision.tool_name, decision.arguments is not None]):
                raise ValueError("Appel d'outil incomplet.")

            result = self.registry.execute(decision.tool_name, decision.arguments)
            state[decision.call_id] = result
            memory.add("tool_result", {
                "call_id": decision.call_id,
                "tool": decision.tool_name,
                "ok": result.ok,
                "value": result.value,
                "error": result.error,
            })

        final = "ARRÊT_BUDGET: nombre maximal d'étapes atteint."
        memory.add("final", final)
        return final, memory, state


agent = ToolLoop(registry, max_steps=6)


## 5. Politique déterministe dépendant des observations

La politique suivante choisit chaque appel à partir de l'état réellement renvoyé. Elle illustre l'orchestration, pas la génération de décisions par un LLM.


In [5]:
def financial_policy(state: dict[str, ToolResult]) -> Decision:
    if "revenue" not in state:
        return Decision(
            "tool_call",
            "Chercher le chiffre d'affaires et sa source.",
            "revenue",
            "recherche_base_connaissances",
            {"requete": "chiffre_affaires_2023"},
        )
    if not state["revenue"].ok:
        return Decision("final", "Source de chiffre d'affaires indisponible.", answer=state["revenue"].error)

    if "headcount" not in state:
        return Decision(
            "tool_call",
            "Chercher l'effectif documenté.",
            "headcount",
            "recherche_base_connaissances",
            {"requete": "effectif_total"},
        )
    if not state["headcount"].ok:
        return Decision("final", "Effectif indisponible.", answer=state["headcount"].error)

    if "usd" not in state:
        revenue = state["revenue"].value
        return Decision(
            "tool_call",
            "Convertir le montant avec le taux pédagogique déclaré.",
            "usd",
            "convertir_devise",
            {
                "montant": revenue["montant"],
                "devise_source": revenue["devise"],
                "devise_cible": "USD",
            },
        )

    if "per_capita" not in state:
        return Decision(
            "tool_call",
            "Calculer le ratio à partir des observations précédentes.",
            "per_capita",
            "calculatrice",
            {
                "a": state["usd"].value["montant"],
                "operation": "/",
                "b": state["headcount"].value["effectif"],
            },
        )

    usd = state["usd"].value["montant"]
    per_capita = state["per_capita"].value
    answer = (
        f"Selon les sources locales du TP, 4 500 000 EUR correspondent à "
        f"{usd:,.2f} USD avec le taux pédagogique, soit {per_capita:,.2f} USD "
        "par collaborateur."
    )
    return Decision("final", "Toutes les valeurs et sources sont disponibles.", answer=answer)


final_answer, financial_memory, financial_state = agent.run(
    "Convertir le chiffre d'affaires 2023 en USD et le rapporter à l'effectif.",
    financial_policy,
)
print(final_answer)
for event in financial_memory.events:
    print(event["kind"], "->", event["content"])

assert financial_state["revenue"].value["source"] == "rapport_annuel_2023"
assert financial_state["per_capita"].ok


Selon les sources locales du TP, 4 500 000 EUR correspondent à 4,891,304.35 USD avec le taux pédagogique, soit 40,760.87 USD par collaborateur.
goal -> Convertir le chiffre d'affaires 2023 en USD et le rapporter à l'effectif.
decision -> Chercher le chiffre d'affaires et sa source.
tool_result -> {'call_id': 'revenue', 'tool': 'recherche_base_connaissances', 'ok': True, 'value': {'montant': 4500000.0, 'devise': 'EUR', 'source': 'rapport_annuel_2023'}, 'error': None}
decision -> Chercher l'effectif documenté.
tool_result -> {'call_id': 'headcount', 'tool': 'recherche_base_connaissances', 'ok': True, 'value': {'effectif': 120, 'source': 'registre_rh_exemple'}, 'error': None}
decision -> Convertir le montant avec le taux pédagogique déclaré.
tool_result -> {'call_id': 'usd', 'tool': 'convertir_devise', 'ok': True, 'value': {'montant': 4891304.347826087, 'devise': 'USD', 'taux': 'constantes_pedagogiques'}, 'error': None}
decision -> Calculer le ratio à partir des observations précédentes.


## 6. Réaction à une erreur réelle

Cette seconde politique tente une division par zéro, lit l'erreur structurée, puis corrige le dénominateur. La correction est réellement déclenchée par l'observation, même si la politique reste déterministe.


In [6]:
def correction_policy(state: dict[str, ToolResult]) -> Decision:
    if "first_try" not in state:
        return Decision(
            "tool_call", "Tester l'appel initial.", "first_try", "calculatrice",
            {"a": 100.0, "operation": "/", "b": 0.0},
        )
    if not state["first_try"].ok and "second_try" not in state:
        return Decision(
            "tool_call", "Corriger le dénominateur après l'erreur.", "second_try", "calculatrice",
            {"a": 100.0, "operation": "/", "b": 4.0},
        )
    if state.get("second_try") and state["second_try"].ok:
        return Decision("final", "Le calcul corrigé est validé.", answer="100 / 4 = 25")
    return Decision("final", "La correction a échoué.", answer="Échec contrôlé")


corrected_answer, correction_memory, correction_state = agent.run(
    "Tester puis corriger une division invalide.", correction_policy
)
print(corrected_answer)
assert correction_state["first_try"].error.startswith("TOOL_ERROR")
assert correction_state["second_try"].value == 25.0


100 / 4 = 25


## 7. Exercice guidé : approbation d'une action à risque

Enregistrez un outil `publier_rapport` avec `risk="high"`. Vérifiez qu'un appel sans approbation est bloqué, puis simulez une approbation explicite.

**Critère de réussite** : la politique du registre, et non le texte produit par un modèle, bloque l'action.


In [7]:
# À vous de jouer.
# TODO : enregistrer l'outil, puis comparer approved=False et approved=True.


In [8]:
@registry.register(risk="high")
def publier_rapport(contenu: str) -> str:
    '''Simule la publication d'un rapport ; aucun système externe n'est modifié.'''
    return f"PUBLICATION_SIMULÉE: {contenu}"


blocked = registry.execute(
    "publier_rapport", {"contenu": "résultats validés"}, approved=False
)
approved = registry.execute(
    "publier_rapport", {"contenu": "résultats validés"}, approved=True
)

assert not blocked.ok and blocked.error == "APPROVAL_REQUIRED"
assert approved.ok and approved.value.startswith("PUBLICATION_SIMULÉE")
print("Sans approbation :", blocked)
print("Avec approbation :", approved)


Sans approbation : ToolResult(ok=False, value=None, error='APPROVAL_REQUIRED')
Avec approbation : ToolResult(ok=True, value='PUBLICATION_SIMULÉE: résultats validés', error=None)


## 8. Tests de robustesse minimaux

Un agent ne se valide pas seulement sur son scénario heureux. Nous testons ici outil inconnu, type invalide et budget d'étapes.


In [9]:
unknown_tool = registry.execute("outil_absent", {})
invalid_type = registry.execute(
    "convertir_devise",
    {"montant": "4500000", "devise_source": "EUR", "devise_cible": "USD"},
)

def endless_policy(_state):
    return Decision(
        "tool_call", "Répéter jusqu'au budget.", "same_call", "calculatrice",
        {"a": 1.0, "operation": "+", "b": 1.0},
    )


bounded_agent = ToolLoop(registry, max_steps=2)
bounded_answer, bounded_memory, _ = bounded_agent.run("Tester le budget.", endless_policy)

assert unknown_tool.error.startswith("UNKNOWN_TOOL")
assert invalid_type.error.startswith("INVALID_TYPE")
assert bounded_answer.startswith("ARRÊT_BUDGET")
print("Outil inconnu :", unknown_tool.error)
print("Type invalide :", invalid_type.error)
print("Boucle bornée :", bounded_answer)


Outil inconnu : UNKNOWN_TOOL: outil_absent
Type invalide : INVALID_TYPE: montant doit être float
Boucle bornée : ARRÊT_BUDGET: nombre maximal d'étapes atteint.


## 9. Où placer un LLM et quand utiliser plusieurs agents ?

Un LLM peut remplacer `financial_policy` en proposant des objets `Decision`. Le registre, les validations, l'approbation et le budget restent dans le programme hôte. Avant branchement, il faut au minimum évaluer sélection d'outil, exactitude des arguments, récupération après erreur, résistance aux contenus non fiables, coût et taux de réussite par scénario.

Plusieurs agents ne sont utiles que si la séparation apporte une compétence, un contexte ou un contrôle réellement distinct. Manager, exécuteur et reviewer partageant les mêmes hypothèses peuvent reproduire la même erreur ; des tests déterministes et des sources indépendantes restent nécessaires.


## Conclusion et auto-évaluation

- Le notebook démontre une **boucle d'orchestration**, pas l'autonomie d'un LLM.
- Les appels d'outils sont structurés, typés et bornés.
- Les actions à risque sont bloquées par du code jusqu'à approbation.
- L'auto-correction est vérifiée sur une erreur réellement renvoyée.
- La prochaine étape serait une évaluation comparative de politiques, jamais l'octroi direct de privilèges au modèle.
